In [ ]:
import os

import torch
import torchani

import Auto3D
from Auto3D import Auto3DOptions
from Auto3D.tautomer import get_stable_tautomers

print(Auto3D.__version__)

# Tautomer Enumeration with Custom NNP

This notebook demonstrates how to use a custom neural network potential for tautomer enumeration.

In [ ]:
# Define a custom NNP. The wrapper must expose coord_pad and species_pad
# attributes and a forward(species, coords, charges) -> energies (eV) method.
class userNNP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Example backend built on ANI2x; replace with your own NNP.
        self.model = torchani.models.ANI2x(periodic_table_index=True)
        self.coord_pad = 0     # padding value for coordinates
        self.species_pad = -1  # padding value for species (atomic numbers)

    def forward(self,
                species: torch.Tensor,
                coords: torch.Tensor,
                charges: torch.Tensor) -> torch.Tensor:
        """species: [B, N] atomic numbers; coords: [B, N, 3]; charges: [B].
        Returns molecular energies [B] in eV."""
        energies = self.model((species, coords)).energies * 27.211386245988
        return energies

In [ ]:
# Initialize, script, and save the custom NNP to a file.
root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
model_path = os.path.join(root, "myNNP.pt")
torch.jit.save(torch.jit.script(userNNP()), model_path)

In [ ]:
input_path = os.path.join(root, "example", "files", "sildnafil.smi")

if __name__ == "__main__":
    config = Auto3DOptions(
        path=input_path, k=1, enumerate_tautomer=True, tauto_engine="rdkit",
        optimizing_engine=model_path,  # path to the saved custom NNP
        max_confs=10, patience=200, use_gpu=False,
    )
    tautomer_out = get_stable_tautomers(config, tauto_k=3)
    print(tautomer_out)